In [1]:
import sys
from pathlib import Path

# Notebook 的工作目录通常是当前文件所在目录，向上查找服务根目录。
service_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
        and (path / "app" / "core" / "config.py").is_file()
    ),
    None,
)
if service_root is None:
    raise RuntimeError("未找到 agent-service 根目录，请从该项目或 notebooks 子目录运行。")

if str(service_root) not in sys.path:
    sys.path.insert(0, str(service_root))


In [10]:
# 1.查询产品及对应的条款
from app.infra.database import AsyncSessionFactory
from app.modules.product.service import ProductService

async with AsyncSessionFactory() as session:
    service = ProductService(session)
    products = await service.list_products(category=None)

for product in products:
    print(product) 

product = products[2]

Product({'category': 'critical_illness', 'id': 1, 'image_url': 'https://files.huizecdn.com/file3/M00/91/12/rBYA3Gi1gZ2AXBL7AAYFJxRhbPg411.png', 'min_premium': Decimal('285.30'), 'target_group': '0-17周岁', 'status': 'active', 'updated_at': '2026-11-12 12:16:08', 'name': '达尔文宝贝计划12号', 'clause_name': '信美相互星辰守护B款少儿重疾险条款.pdf', 'insurer': '信美人寿相互保险社', 'description': '117种重疾+28种中症+45种轻症，少儿重疾保障灵活。', 'max_premium': Decimal('2501.83'), 'highlights': ['117种重疾', '少儿保障', '可选投保人豁免'], 'created_at': '2026-11-12 12:16:08'})
Product({'category': 'critical_illness', 'id': 2, 'image_url': 'https://files.huizecdn.com/file3/M00/90/E5/rBYA3Gi1BnWALIBtAAYFAsRjzwk030.png', 'min_premium': Decimal('561.00'), 'target_group': '28天-35周岁（1-4类职业）', 'status': 'active', 'updated_at': '2026-11-12 12:16:08', 'name': '达尔文12号重大疾病保险', 'clause_name': '复星联合优选K重疾险(达尔文12号)条款.pdf', 'insurer': '复星联合健康保险', 'description': '120种重疾+30种中症+45种轻症，可选多项责任。', 'max_premium': Decimal('4815.00'), 'highlights': ['120种重疾', '中轻症保障', '可选责任丰富'], 'c

In [5]:
# 2.解析文档
from app.core.config import settings
from mineru import MinerU

# 创建客户端
client = MinerU(token=settings.rag.mineru_token)
# 解析
result = client.extract(f"../../data/raw/kb/{product.clause_name}")

In [6]:
# 保存解析结果
result.save_markdown('output/r1.md')

PosixPath('output/r1.md')

In [7]:
# 3.文档优化，确保文档标题正确解析
from langchain.chat_models import init_chat_model

MARKDOWN_OPTIMIZATION_PROMPT = """
下面这份Markdown文档是从保险条款PDF解析得来，由于PDF中的各个小节是以表格形式存在，所以解析时出现错乱。
你分析内容，帮我转为格式正确的Markdown，特别是标题编号要正确。
- 标题等级要从1级标题开始，逐层增加，目录和文档名不计入标题等级。
- 输出结果中不要包含正文开始之前的部分。
- 输出结果只包含Markdown正文，不要解释或代码围栏。
- 不要修改保险条款。
""".strip()

llm = init_chat_model(
    "deepseek-flash",
    api_key=settings.llm.api_key,
    extra_body={'thinking': {'type': 'disabled'}},
    max_tokens=10000
)

response = llm.invoke([
    {"role": "system", "content": MARKDOWN_OPTIMIZATION_PROMPT},
    {"role": "user", "content": result.markdown},
])

if isinstance(response.content, str):
    markdown = response.content

OSError: [Errno 30] Read-only file system: '/output'

In [8]:
# 保存文档
output_path = Path("output/r2.md")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(markdown, encoding="utf-8")

17540

In [9]:
from app.rag.models import ParentChunk
from langchain_classic.schema import Document
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

# 4.文档切分
# 4.1.markdown切分器，处理父块
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
        ("####", "Header 4"),
        ("#####", "Header 5"),
        ("######", "Header 6"),
    ],
    strip_headers=True,
)

# 4.2.递归切分器，处理子块
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", "。", "；", ";", "，", ","],
    keep_separator="end",
)

# 4.3.先用 markdown切分，再用递归切分
children_texts = markdown_splitter.split_text(markdown)
children_texts[0]

Document(metadata={'Header 1': '复星联合医联有盟重大疾病保险（互联网）条款'}, page_content='（本公司在每页底部对一些专业名词做了释义，这些释义为本条款的重要组成部分。同一专业名词在条款中出现多次的，本公司仅在该专业名词第一次出现的地方做了释义，该释义适用于全文。）')

In [11]:
# 4.4.处理子块
import uuid

parent_chunks: list[ParentChunk] = []
child_chunks: list[Document] = []

for section in children_texts:
    # 生成章节路径，section.metadata 就是每一级标题的字典
    session_path = list(section.metadata.values())
    # 组织 parent_chunk
    parent_chunk = ParentChunk(
        id = uuid.uuid4(),
        product_id = product.id,
        content=section.page_content,
        section_path=session_path,
        clause_name=product.clause_name
    )
    parent_chunks.append(parent_chunk)

    # 组织 section 的 metadata
    section.metadata = {
        "product_id": product.id,
        "parent_id": str(parent_chunk.id),
    }

    # 切分 section 得到子块
    children = text_splitter.split_documents([section])
    # 把父块的 section_path添加到子块的 metadata 中
    section_path_header = "\n".join(session_path) + "\n"
    for child in children:
        child.page_content = section_path_header + child.page_content
    child_chunks.extend(children)

In [13]:
# 5.子块进入向量数据库
from langchain_milvus import BM25BuiltInFunction, Milvus
from langchain_ollama import OllamaEmbeddings

# 创建向量模型，可以把文档转稠密向量
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024,  # 可选：减少维度以节省存储
    base_url="http://10.40.0.31:11434",  # 可选：指定 Ollama 服务器地址
)

# 初始化 vector store
vector_store = Milvus(
    embedding_function=embeddings,
    collection_name="insurance_collection",
    builtin_function=BM25BuiltInFunction(analyzer_params={"type": "chinese"}),
    vector_field=["dense", "sparse"],
    connection_args={"uri": settings.rag.milvus_url},
    auto_id=True,
)

# 删除旧文档
vector_store.delete(expr=f"product_id=={product.id}")

# 添加新文档
vector_store.add_documents(child_chunks)

2026-11-15 19:19:10,256 [ERROR][_log_rpc_error]: RPC error: [delete], <MilvusException: (code=100, message=Failed to get collection id: collection not found[database=default][collection=insurance_collection])>, <elapsed:5.0ms>
Traceback:
Traceback (most recent call last):
  File "/Users/myl/Documents/python-code/agent-projects/hm-insurance/agent-service/.venv/lib/python3.14/site-packages/pymilvus/decorators.py", line 518, in handler
    return func(*args, **kwargs)
  File "/Users/myl/Documents/python-code/agent-projects/hm-insurance/agent-service/.venv/lib/python3.14/site-packages/pymilvus/decorators.py", line 565, in handler
    return func(self, *args, **kwargs)
  File "/Users/myl/Documents/python-code/agent-projects/hm-insurance/agent-service/.venv/lib/python3.14/site-packages/pymilvus/decorators.py", line 456, in handler
    raise e from e
  File "/Users/myl/Documents/python-code/agent-projects/hm-insurance/agent-service/.venv/lib/python3.14/site-packages/pymilvus/decorators.py", l

[470413685495124360,
 470413685495124361,
 470413685495124362,
 470413685495124363,
 470413685495124364,
 470413685495124365,
 470413685495124366,
 470413685495124367,
 470413685495124368,
 470413685495124369,
 470413685495124370,
 470413685495124371,
 470413685495124372,
 470413685495124373,
 470413685495124374,
 470413685495124375,
 470413685495124376,
 470413685495124377,
 470413685495124378,
 470413685495124379,
 470413685495124380,
 470413685495124381,
 470413685495124382,
 470413685495124383,
 470413685495124384,
 470413685495124385,
 470413685495124386,
 470413685495124387,
 470413685495124388,
 470413685495124389,
 470413685495124390,
 470413685495124391,
 470413685495124392,
 470413685495124393,
 470413685495124394,
 470413685495124395,
 470413685495124396,
 470413685495124397,
 470413685495124398,
 470413685495124399,
 470413685495124400,
 470413685495124401,
 470413685495124402,
 470413685495124403,
 470413685495124404,
 470413685495124405,
 470413685495124406,
 470413685495

In [14]:
# 6.父知识块入库

from app.rag.repository import ParentChunkRepository

async with AsyncSessionFactory() as session, session.begin():
    repository = ParentChunkRepository(session)
    # 先删除对应的父块内容
    await repository.delete_by_product_id(product_id=product.id)
    # 批量全部添加
    repository.add_all(parent_chunks)